In [14]:
import os
import mujoco
import mujoco.viewer
import numpy as np

In [5]:
model = mujoco.MjModel.from_xml_path('humanoid.xml')

In [75]:
import xml.etree.ElementTree as ET

def create_mjcf_humanoid(positions):
    if len(positions) != 10:
        raise ValueError("There must be exactly 10 positions provided.")

    # Helper function to create a body with a geom
    def create_body(parent, name, pos, size, rgba, geom_type="sphere", joint_type="hinge"):
        body = ET.SubElement(parent, 'body', name=name, pos=" ".join(map(str, pos)))
        ET.SubElement(body, 'joint', name=name, type=joint_type, pos=" ".join(map(str, pos)))
        ET.SubElement(body, 'geom', type=geom_type, size=" ".join(map(str, size)), rgba=" ".join(map(str, rgba)))
        return body

    # Create the root element
    model = ET.Element('mujoco', model="17_part_humanoid")
    ET.SubElement(model, 'compiler', angle="radian")
    ET.SubElement(model, 'option', timestep="0.002")

    # Create the worldbody element
    worldbody = ET.SubElement(model, 'worldbody')

    # Extract positions
    torso_pos = positions[0]
    head_pos = positions[1]
    right_arm_pos = positions[2]
    right_forearm_pos = positions[3]
    left_arm_pos = positions[4]
    left_forearm_pos = positions[5]
    right_leg_pos = positions[6]
    right_calf_pos = positions[7]
    left_leg_pos = positions[8]
    left_calf_pos = positions[9]

    # Create the torso and subparts
    torso = create_body(worldbody, "torso", torso_pos, [0.2, 0.1, 0.3], [0.6, 0.6, 0.6, 1])
    
    # Head
    create_body(torso, "head", head_pos, [0.1], [0.8, 0.6, 0.6, 1], geom_type="sphere")
    
    # Right Arm and Forearm
    right_arm = create_body(torso, "right_arm", right_arm_pos, [0.05, 0.05, 0.2], [0.6, 0.6, 0.8, 1])
    create_body(right_arm, "right_forearm", right_forearm_pos, [0.05, 0.05, 0.2], [0.6, 0.6, 0.8, 1])
    
    # Left Arm and Forearm
    left_arm = create_body(torso, "left_arm", left_arm_pos, [0.05, 0.05, 0.2], [0.6, 0.6, 0.8, 1])
    create_body(left_arm, "left_forearm", left_forearm_pos, [0.05, 0.05, 0.2], [0.6, 0.6, 0.8, 1])
    
    # Right Leg and Calf
    right_leg = create_body(torso, "right_leg", right_leg_pos, [0.1, 0.1, 0.3], [0.8, 0.6, 0.6, 1])
    create_body(right_leg, "right_calf", right_calf_pos, [0.1, 0.1, 0.3], [0.8, 0.6, 0.6, 1])
    
    # Left Leg and Calf
    left_leg = create_body(torso, "left_leg", left_leg_pos, [0.1, 0.1, 0.3], [0.8, 0.6, 0.6, 1])
    create_body(left_leg, "left_calf", left_calf_pos, [0.1, 0.1, 0.3], [0.8, 0.6, 0.6, 1])

    # Create joints (simple example, they are free joints)
    joints = [
        ("torso_joint", torso_pos),
        ("head_joint", head_pos),
        ("right_arm_joint", right_arm_pos),
        ("right_forearm_joint", right_forearm_pos),
        ("left_arm_joint", left_arm_pos),
        ("left_forearm_joint", left_forearm_pos),
        ("right_leg_joint", right_leg_pos),
        ("right_calf_joint", right_calf_pos),
        ("left_leg_joint", left_leg_pos),
        ("left_calf_joint", left_calf_pos),
    ]
    for name, pos in joints:
        ET.SubElement(model, 'joint', name=name, type="free", pos=" ".join(map(str, pos)))

    # Add actuators
    actuators = [
        "torso_joint",
        "head_joint",
        "right_arm_joint",
        "right_forearm_joint",
        "left_arm_joint",
        "left_forearm_joint",
        "right_leg_joint",
        "right_calf_joint",
        "left_leg_joint",
        "left_calf_joint"
    ]
    actuator = ET.SubElement(model, 'actuator')
    for joint in actuators:
        ET.SubElement(actuator, 'motor', joint=joint, ctrlrange="-1 1")

    # Convert the ElementTree to a string
    tree = ET.ElementTree(model)
    from io import StringIO
    output = StringIO()
    tree.write(output, encoding='unicode', xml_declaration=True)
    return output.getvalue()


def create_custom_humanoid(positions):
    def create_torso(parent, name, pos_torso, pos_head, pos_neck, rgba=[0.8, 0.6, 0.6, 1]):
        body = ET.SubElement(parent, 'body', name=name, pos=" ".join(map(str, pos_torso)))
        ET.SubElement(body, 'camera', mode="trackcom", pos="1 -4 0", xyaxes="1 0 0 0 0 1")
        ET.SubElement(body, 'geom', type="capsule", size=".07", rgba=" ".join(map(str, rgba)))
        ET.SubElement(body, 'geom', name="head", pos=" ".join(map(str, pos_head)), size=".08", 
                      type="sphere", rgba=" ".join(map(str, rgba)))
        ET.SubElement(body, 'geom', name="neck", pos=" ".join(map(str, pos_neck)), size=".04", 
                      type="sphere", rgba=" ".join(map(str, rgba)))
        ET.SubElement(body, 'joint', name="root", pos="0 0 0", 
                      type="free", armature="0", damping="0", stiffness="0", limited="false")

        return body
        
    def create_abdomen(parent, name, pos_lwaist, rgba=[0.8, 0.6, 0.6, 1]):
        body = ET.SubElement(parent, 'body', name=name, pos=" ".join(map(str, pos_lwaist)),
                            quat="1.000 0 -0.002 0")

        ET.SubElement(body, 'joint', name="abdomen_z", pos="0 0 0.065", axis="0 0 1",
                      type="hinge", armature="0.02", damping="5", stiffness="20")
        ET.SubElement(body, 'joint', name="abdomen_y", pos="0 0 0.065", axis="0 1 0",
                      type="hinge", armature="0.02", damping="5", stiffness="10")

        return body
        
    def create_pelvis(parent, name, pos_pelvis, rgba=[0.8, 0.6, 0.6, 1]):
        body = ET.SubElement(parent, 'body', name=name, pos=" ".join(map(str, pos_pelvis)))
        
        from_ = pos_pelvis + np.array([-.01,-.07,0])
        to = pos_pelvis + np.array([-.01,.07,0])
        fromto = " ".join(map(str, from_)) + " " + " ".join(map(str, to))
        ET.SubElement(body, 'geom', type="capsule", size="0.07", 
                      fromto=fromto, rgba=" ".join(map(str, rgba)))
        ET.SubElement(body, 'joint', name="abdomen_x", pos="0 0 0.1", axis="1 0 0",
                      type="hinge", armature="0.02", range="-35 35", damping="5", stiffness="10")    

        return body
        
    def create_leg(parent, name, pos_thigh, pos_shin, pos_foot, rgba=[0.8, 0.6, 0.6, 1], side="left"):
        body = ET.SubElement(parent, 'body', name=name, pos=" ".join(map(str, pos_thigh)))
        
        d = {"left": -1, "right": 1}
        
        ET.SubElement(body, 'joint', name=side + "_hip_x", pos="0 0 0", axis=str(d[side]) + " 0 0",
                      type="hinge", armature="0.01", range="-25 5", damping="5", stiffness="10") 
        ET.SubElement(body, 'joint', name=side + "_hip_z", pos="0 0 0", axis="0 0 " + str(d[side]),
                      type="hinge", armature="0.01", range="-60 35", damping="5", stiffness="10") 
        ET.SubElement(body, 'joint', name=side + "_hip_y", pos="0 0 0", axis="0 1 0",
                      type="hinge", armature="0.01", range="-110 20", damping="5", stiffness="20") 
    
        fromto_thigh = " ".join(map(str,pos_thigh)) + " " +  " ".join(map(str,pos_shin))
        ET.SubElement(body, 'geom', type="capsule", name=side + "_thigh",
                      fromto=fromto_thigh, size="0.06", rgba=" ".join(map(str, rgba)))
        
        shin = ET.SubElement(body, 'body', name=side+"_shin", pos=" ".join(map(str, pos_shin)))
        fromto_shin = " ".join(map(str,pos_shin)) + " " +  " ".join(map(str,pos_foot))

        ET.SubElement(shin, 'joint', name=side + "_knee", pos="0 0 0", axis=str(d[side]) + " 0 0",
                      type="hinge", armature="0.006", range="-160 -2", damping="5", stiffness="10") 
        ET.SubElement(shin, 'geom', fromto=fromto_shin, name=side+"_shin1", size="0.049", 
                      pos=" ".join(map(str, pos_foot)))
        foot = ET.SubElement(shin, 'body', name=side+"_foot", pos=" ".join(map(str, pos_foot)))
        ET.SubElement(foot, 'geom', type="sphere", name=side + "_foot", pos="0 0 0.1",
                      size="0.07", rgba=" ".join(map(str, rgba)))
        
        return body, shin, foot

        
    def create_arm(parent, name, pos_shoulder, pos_elbow, pos_hand, rgba=[0.8, 0.6, 0.6, 1], side="left"):
        body = ET.SubElement(parent, 'body', name=name, pos=" ".join(map(str, pos_shoulder)))
        
        d = {"left": -1, "right": 1}

        ET.SubElement(body, 'joint', name=side + "_shoulder1", pos="0 0 0", axis="2 " + str(d[side]) + " 1",
                      type="hinge", armature="0.0068", range="-85 60", stiffness="1") 
        ET.SubElement(body, 'joint', name=side + "_shoulder2", pos="0 0 0", axis="0 " + str(d[side]*-1) + " 1",
                      type="hinge", armature="0.0051", range="-85 60", stiffness="1") 
    
        fromto_upper = " ".join(map(str, pos_shoulder)) + " " +  " ".join(map(str, pos_elbow))
        ET.SubElement(body, 'geom', type="capsule", name=side + "_upper_arm", fromto=fromto_upper,
                      size="0.04 0.16", rgba=" ".join(map(str, rgba)))     
        
        lower_arm = ET.SubElement(body, 'body', name=side+"_lower_arm", pos=" ".join(map(str, pos_elbow)))
        fromto_lower = " ".join(map(str, pos_elbow)) + " " +  " ".join(map(str, pos_hand))
        ET.SubElement(lower_arm, 'geom', type="capsule", name=side + "_larm", fromto=fromto_lower,
                      size="0.031", rgba=" ".join(map(str, rgba)))     
        ET.SubElement(lower_arm, 'joint', name=side + "_elbow", pos="0 0 0", axis="0 -1 " + str(d[side]),
                      type="hinge", armature="0.0028", range="-90 50", stiffness="1")         
        ET.SubElement(lower_arm, 'geom', type="sphere", name=side + "_hand", pos=" ".join(map(str, pos_hand)),
                      size="0.04", rgba=" ".join(map(str, rgba)))     
        
        return body, lower_arm
        
    def create_tendons(parent):
        tendon = ET.SubElement(parent, 'tendon')
        left_hipknee = ET.SubElement(tendon, 'fixed', name="left_hipknee")
        ET.SubElement(left_hipknee, 'joint', coef="-1", joint="left_hip_y")
        ET.SubElement(left_hipknee, 'joint', coef="1", joint="left_knee")
        right_hipknee = ET.SubElement(tendon, 'fixed', name="right_hipknee")
        ET.SubElement(right_hipknee, 'joint', coef="-1", joint="right_hip_y")
        ET.SubElement(right_hipknee, 'joint', coef="1", joint="right_knee")

        
    # take absolute positions from dict and compute relative positions according to limb hierarchy
    pos_torso = positions["torso"]
    pos_head = positions["head"] - pos_torso
    pos_neck = positions["neck"] - pos_torso
    
    pos_abdomen = positions["abdomen"] - pos_torso
    pos_pelvis = positions["pelvis"] - pos_abdomen
    
    pos_right_thigh = positions["right_thigh"] - pos_pelvis
    pos_right_knee = positions["right_knee"] - pos_right_thigh
    pos_right_foot = positions["right_foot"] - pos_right_knee
    
    pos_left_thigh = positions["left_thigh"] - pos_pelvis
    pos_left_knee = positions["left_knee"] - pos_left_thigh
    pos_left_foot = positions["left_foot"] - pos_left_knee
    
    pos_left_shoulder = positions["left_shoulder"] - pos_torso
    pos_left_elbow = positions["left_elbow"] - pos_left_shoulder
    pos_left_hand = positions["left_hand"] - pos_left_elbow

    pos_right_shoulder = positions["right_shoulder"] - pos_torso
    pos_right_elbow = positions["right_elbow"] - pos_right_shoulder
    pos_right_hand = positions["right_hand"] - pos_right_elbow
    
    ### create the parts 
    
    # Create the root element
    model = ET.Element('mujoco', model="custom_humanoid")
    ET.SubElement(model, 'compiler', angle="radian")
    ET.SubElement(model, 'option', timestep="0.002")

    # Create the worldbody element
    worldbody = ET.SubElement(model, 'worldbody')

    torso = create_torso(worldbody, "torso", pos_torso, pos_head, pos_neck)

    abdomen = create_abdomen(torso, "abdomen", pos_abdomen)
    pelvis = create_pelvis(abdomen, "pelvis", pos_pelvis)
    lleg = create_leg(pelvis, "left_leg", pos_left_thigh, pos_left_knee, pos_left_foot, side="left")
    rleg = create_leg(pelvis, "right_leg", pos_right_thigh, pos_right_knee, pos_right_foot, side="right")
    larm = create_arm(torso, "left_arm", pos_left_shoulder, pos_left_elbow, pos_left_hand, side="left")
    rarm = create_arm(torso, "right_arm", pos_right_shoulder, pos_right_elbow, pos_right_hand, side="right")
    create_tendons(model)
    
    # Create actuators for the joints
    actuated_joints = ["abdomen_x", "abdomen_y", "abdomen_z",
                      "right_hip_x", "right_hip_z", "right_hip_y",
                      "right_knee", "left_hip_x", "left_hip_z",
                      "left_hip_y", "left_knee", "right_shoulder1",
                      "right_shoulder2", "right_elbow", "left_shoulder1",
                      "left_shoulder2", "left_elbow"]

    actuator = ET.SubElement(model, 'actuator')
    for joint in actuated_joints:
        ET.SubElement(actuator, 'motor', joint=joint, ctrlrange="-1 1")
       
    # Convert the ElementTree to a string
    tree = ET.ElementTree(model)
    ET.indent(tree, space="\t", level=0)
    from io import StringIO
    output = StringIO()
    tree.write(output, encoding='unicode', xml_declaration=True)
    return output.getvalue()
        
#element = ET.XML("<html><body>text</body></html>")
#ET.indent(element)
#print(ET.tostring(element, encoding='unicode'))

In [76]:
# Example usage:
positions = [
    (0, 0, 1),    # torso
    (0, 0, 1.35), # head
    (0.25, 0, 0.1), # right arm
    (0.25, 0, -0.1), # right forearm
    (-0.25, 0, 0.1), # left arm
    (-0.25, 0, -0.1), # left forearm
    (0.1, 0, 0.0), # right leg
    (0.1, 0, -0.3), # right calf
    (-0.1, 0, 0.0), # left leg
    (-0.1, 0, -0.3), # left calf
    # Add more body parts if needed
]

#mjcf_description = create_mjcf_humanoid(positions)
#print(mjcf_description)

positions_dict = {"torso": [0, 0, 1.4],
                  "head": [0.,0.,1.59], #0 0 .19
                  "neck": [0.,0.,1.5], # 0 0 .1
                  "abdomen": [-.01, 0, 1.25], # -.01 0 -0.15
                  "pelvis": [-.01, 0, 1.085], # 0 0 -0.165
                  "right_thigh": [-.01, -0.1, 1.045], #0 -0.1 -0.04
                  "right_knee": [-.01,-0.09, 0.642], # 0 0.01 -0.403
                  "right_foot": [-.01,-0.09, 0.192], #0 0 -0.45
                  "left_thigh": [-.01, 0.1, 1.045], #0 0.1 -0.04
                  "left_knee": [-.01,0.09, 0.642], # 0 0.01 -0.403
                  "left_foot": [-.01,0.09, 0.192], #0 0 -0.45
                  "right_shoulder": [0, -.17, 1.46], #0 -0.17 0.06
                  "right_elbow": [.18, -.35, 1.28], #.18 -.18 -.18
                  "right_hand": [.36, -.17, 1.46], # .18 .18 .18
                  "left_shoulder": [0, .17, 1.46], #0 0.17 0.06
                  "left_elbow": [.18, .35, 1.28], #.18 .18 -.18
                  "left_hand": [.36, .17, 1.46] # .18 -.18 .18
                 }

for k,v in positions_dict.items():
    positions_dict[k] = np.array(v)

mjcf_description = create_custom_humanoid(positions_dict)
print(mjcf_description)

<?xml version='1.0' encoding='utf-8'?>
<mujoco model="custom_humanoid">
	<compiler angle="radian" />
	<option timestep="0.002" />
	<worldbody>
		<body name="torso" pos="0.0 0.0 1.4">
			<camera mode="trackcom" pos="1 -4 0" xyaxes="1 0 0 0 0 1" />
			<geom type="capsule" size=".07" rgba="0.8 0.6 0.6 1" />
			<geom name="head" pos="0.0 0.0 0.19000000000000017" size=".08" type="sphere" rgba="0.8 0.6 0.6 1" />
			<geom name="neck" pos="0.0 0.0 0.10000000000000009" size=".04" type="sphere" rgba="0.8 0.6 0.6 1" />
			<joint name="root" pos="0 0 0" type="free" armature="0" damping="0" stiffness="0" limited="false" />
			<body name="abdomen" pos="-0.01 0.0 -0.1499999999999999" quat="1.000 0 -0.002 0">
				<joint name="abdomen_z" pos="0 0 0.065" axis="0 0 1" type="hinge" armature="0.02" damping="5" stiffness="20" />
				<joint name="abdomen_y" pos="0 0 0.065" axis="0 1 0" type="hinge" armature="0.02" damping="5" stiffness="10" />
				<body name="pelvis" pos="0.0 0.0 1.2349999999999999">
					<g

In [77]:
m = mujoco.MjModel.from_xml_string(mjcf_description)
d = mujoco.MjData(m)

mujoco.viewer.launch(m)

ValueError: Error: sizes must be positive in object '' (id = 0)
Object name = , id = 0, line = 8, column = -1